# Data Setup Guide — Financial Machine Learning

**Companion notebook for the Colab labs (Weeks 1-16).** For every lab, this notebook gives you a
runnable way to get the data: the free `yfinance` library as the main path for market prices,
Kaggle for fraud/credit datasets, and fully offline simulated fallbacks if the internet is blocked.

### How to use
1. Run the **Setup** cell once.
2. Jump to the section for the week you are working on and run its loader cell.
3. Each loader returns clean `pandas` / `numpy` objects ready for the lab tasks.

> **Tip:** download once, then save to CSV (shown below) so re-running the lab is instant and you
> avoid Yahoo rate limits.


---
## 0. Setup - run this first


In [ ]:
# Colab already has pandas/numpy/scikit-learn. Install the rest quietly.
!pip -q install yfinance scipy 2>/dev/null

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option('display.max_rows', 12)
np.random.seed(42)            # reproducible simulations
print('Setup complete.')

---
## 1. `yfinance` primer (the main path for prices)

`yfinance` pulls historical prices straight from Yahoo Finance - **free, no API key, works in Colab.**

### One important recent change
Since yfinance 0.2.28+, `yf.download()` defaults to **`auto_adjust=True`**. That means:
- the returned **`Close` column is already the *adjusted* close** (splits/dividends handled), and
- the old separate `Adj Close` column **no longer exists** by default.

For return modeling the adjusted close is exactly what we want, so we keep `auto_adjust=True`.


In [ ]:
import yfinance as yf

# --- single ticker ---
# multi_level_index=False keeps the columns flat (Open/High/Low/Close/Volume).
aapl = yf.download('AAPL', start='2022-01-01', end='2024-12-31',
                   auto_adjust=True, multi_level_index=False, progress=False)
print(aapl.shape)
aapl[['Close', 'Volume']].head()

In [ ]:
# 'Close' is already the ADJUSTED close. Build returns from it:
px         = aapl['Close'].dropna()
log_ret    = np.log(px).diff().dropna()      # daily log returns
simple_ret = px.pct_change().dropna()        # simple returns

print(f'{len(log_ret)} daily returns')
print(f'annualized vol = {log_ret.std() * np.sqrt(252):.3f}')
log_ret.head()

In [ ]:
# --- several tickers at once -> a DataFrame of adjusted closes ---
panel = yf.download(['AAPL', 'MSFT', 'KO', 'PEP'],
                    start='2022-01-01', end='2024-12-31',
                    auto_adjust=True, progress=False)['Close']
panel.tail()

In [ ]:
# --- CACHE to avoid re-downloading (and rate limits) ---
panel.to_csv('prices_cache.csv')

# later, reload instantly:
# panel = pd.read_csv('prices_cache.csv', index_col=0, parse_dates=True)
print('saved prices_cache.csv')

> **If you hit `YFRateLimitError: Too Many Requests`:** wait a minute, request fewer tickers,
> or load from your cached CSV. Yahoo throttles heavy use.

**Handy tickers:** `^GSPC` (S&P 500), `^IXIC` (Nasdaq), `^KS11` (KOSPI), `^KQ11` (KOSDAQ),
`AAPL`/`MSFT`/... (US stocks), `005930.KS` (Samsung Elec.), `BTC-USD`, `EURUSD=X`,
`^VIX` (volatility), `^TNX` (10Y yield).


---
## 2. Week 1 - Returns & the Bias-Variance Idea

**Labs need:** (a) ~2 years of daily prices for one stock, turned into returns to inspect normality;
(b) a small noisy dataset to demonstrate overfitting.


In [ ]:
# (a) Week 1 Lab 1 data: one ticker -> returns -> normality check
w1_px  = yf.download('AAPL', period='2y', auto_adjust=True,
                     multi_level_index=False, progress=False)['Close'].dropna()
w1_ret = w1_px.pct_change().dropna()

print('skewness        :', round(stats.skew(w1_ret), 3))
print('excess kurtosis :', round(stats.kurtosis(w1_ret), 3), '  (>0 means fat tails)')

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(w1_ret, bins=60); ax[0].set_title('Daily returns')
stats.probplot(w1_ret, dist='norm', plot=ax[1])
plt.tight_layout(); plt.show()

In [ ]:
# (b) Week 1 Lab 2 data: noisy curve to SEE overfitting
x = np.linspace(0, 1, 30)
y = np.sin(2 * np.pi * x) + np.random.normal(0, 0.2, 30)

xs = np.linspace(0, 1, 200)
for d in [1, 4, 15]:                      # under / good / over
    coef = np.polyfit(x, y, d)
    plt.plot(xs, np.polyval(coef, xs), label=f'degree {d}')
plt.scatter(x, y, color='black', s=20)
plt.legend(); plt.ylim(-2, 2); plt.title('Bias-Variance: watch degree 15 wiggle'); plt.show()

---
## 3. Weeks 2-4 - Bars, Labeling, and a Multi-Asset Panel

**Labs need:** intraday-style data for Dollar Bars (we approximate with daily price x volume),
and a panel of returns for the linear-model / index-tracking lab.


In [ ]:
# (a) approximate 'dollar volume' for the Bars lab (Week 2)
bars_raw = yf.download('AAPL', period='1y', auto_adjust=True,
                       multi_level_index=False, progress=False)
bars_raw['dollar_vol'] = bars_raw['Close'] * bars_raw['Volume']
print('total dollar volume over the year: ${:,.0f}'.format(bars_raw['dollar_vol'].sum()))
bars_raw[['Close', 'Volume', 'dollar_vol']].head()

In [ ]:
# (b) a panel of daily returns for Weeks 4-9 (linear models, PCA, portfolios)
tickers = ['AAPL','MSFT','GOOGL','AMZN','META','KO','PEP','XOM','JPM','JNJ']
prices  = yf.download(tickers, start='2020-01-01', end='2024-12-31',
                      auto_adjust=True, progress=False)['Close'].dropna()
rets    = prices.pct_change().dropna()
print('panel shape (days x assets):', rets.shape)
rets.head()

---
## 4. Weeks 5-8 - Classification Datasets (Fraud & Default)

These problems need labeled tabular data we cannot get from prices. We use **Kaggle**.

### How to load a Kaggle dataset in Colab
1. On kaggle.com: **Account -> Settings -> Create New API Token**. This downloads `kaggle.json`.
2. Run the cell below and upload that file when prompted.
3. Then download any dataset by its slug.


In [ ]:
# --- one-time Kaggle auth in Colab ---
from google.colab import files
files.upload()                                    # choose your kaggle.json

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!pip -q install kaggle
print('Kaggle ready.')

In [ ]:
# --- Credit Card Fraud dataset (Weeks 5-8 fraud labs) ---
# 284,807 transactions, only 0.17% fraud -> classic imbalance.
!kaggle datasets download -d mlg-ulb/creditcardfraud
!unzip -o creditcardfraud.zip

fraud = pd.read_csv('creditcard.csv')
print('shape:', fraud.shape)
print('fraud rate: {:.3%}'.format(fraud['Class'].mean()))
fraud.head()

In [ ]:
# --- 'Give Me Some Credit' default dataset (Week 5/7 default prediction) ---
# alternative to Home Credit; smaller and quick to load.
!kaggle competitions download -c GiveMeSomeCredit  2>/dev/null || \
  echo 'If this fails, accept the competition rules once on kaggle.com, then re-run.'
# !unzip -o GiveMeSomeCredit.zip
print('See note above if download was blocked.')

---
## 5. Offline fallback - no internet needed

If Yahoo and Kaggle are blocked, every lab can still run on **simulated data**.
`scikit-learn` can generate a realistic imbalanced classification set that behaves like fraud.


In [ ]:
from sklearn.datasets import make_classification

# fraud-like imbalanced data (1% positive) for Weeks 5-8
X, yb = make_classification(n_samples=20000, n_features=20, n_informative=6,
                            weights=[0.99, 0.01], random_state=42)
print('class balance:', np.bincount(yb), ' -> ~1% positive (fraud-like)')

# noisy regression data for Weeks 4-8 (bias-variance, linear models)
x_sim = np.linspace(0, 1, 200)
y_sim = np.sin(2*np.pi*x_sim) + np.random.normal(0, 0.2, 200)

# a random-walk 'price' + its returns for Weeks 1-2
price_sim = 100 * np.exp(np.cumsum(np.random.normal(0, 0.01, 1000)))
ret_sim   = pd.Series(price_sim).pct_change().dropna()
print('Offline datasets ready: X/yb (fraud-like), x_sim/y_sim (noisy curve), price_sim/ret_sim.')

---
## 6. Quick map - which section for which lab

| Week | Lab topic | Run section |
|---|---|---|
| 1 | Returns & normality; overfitting demo | Section 2 |
| 2 | Dollar Bars; CUSUM events | Section 3 (a) |
| 3 | Triple-Barrier labeling | Section 3 (b) |
| 4 | Linear models; index tracking | Section 3 (b) |
| 5-8 | Fraud / default classification | Section 4 (or Section 5 offline) |
| 9 | PCA, denoising, clustering | Section 3 (b) |
| any | No internet available | Section 5 offline fallback |

All loaders return tidy `pandas` / `numpy` objects ready to plot, test, and model.
